# Resnet18 Contrastive Tags

This notebook trains a late-fusion audio encoder for content-based song retrieval.

The idea is simple:
- notebook 2 provides a frozen tag embedding space,
- each song keeps its set of valid tag vectors,
- the audio model is trained so similar tag sets lead to similar song embeddings.


## Notes
- Train on `data/processed/train.parquet`, retrieve on `data/processed/test.parquet`.
- The semantic teacher comes directly from notebook 2 tag vectors.
- Baseline retrieval is plain cosine similarity on the learned `song_embedding`.
- There is no train/test leakage in this notebook: optimization uses only `train_df`, and any test-side tag processing is used only for held-out analysis after training.


## Flow

1. Load the processed train/test splits.
2. Convert each song's clean tags into a frozen tag set from notebook 2.
3. Build mix + stem spectrogram batches.
4. Encode audio with a shared ResNet18.
5. Train the fused song embedding with tag-set similarity and light audio regularization.
6. Encode the test split and retrieve neighbors by cosine similarity.
7. Save the run artifacts for reuse.


## System Flowchart (High Level)

```text
[Song spectrograms: mix + stems]      [Frozen notebook 2 tag vectors]
              |                                      |
              v                                      v
       [Shared ResNet18]                    [Per-song tag sets]
          /          \                                |
         v            v                               v
 [Mix embedding]  [Stem embeddings]      [Batch tag-similarity matrix]
                     /      \
                    v        v
          [Harmonic path] [Drum path]
             (bass/other/vocals)
                  \      /
                   v    v
         [Late Fusion: mix + a*harmony + b*drums]
                           |
                           v
                    [Song embedding]
                      /         \
                     v           v
    [Relational tag loss on batch similarities] [Cosine retrieval]
         + small same-song InfoNCE stabilizer       (top-k recs)
```


## 1) Environment Setup

Import the libraries and project helpers used throughout the notebook.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
import random
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from torchvision.models import ResNet18_Weights, resnet18


ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists() and ROOT_DIR.parent.exists():
    ROOT_DIR = ROOT_DIR.parent
SRC_DIR = ROOT_DIR / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from song_recommender.paths import DATA_DIR, TAG_KEYS, TAG_VECTORS, VALID_TAGS
from song_recommender.data.indexer import TrackIndexer
from song_recommender.data.loader import load_png_resized
from song_recommender.features.tag_features import clean_tags


## 2) Hyperparameters (Main Control Panel)

Most experiment changes happen in `CFG`. Keep the notebook logic fixed and edit this cell when you want a new run.


## 2b) What Changed From Notebook 4

- Kept the same architecture and data pipeline.
- Extended training to 80 epochs to match the stronger notebook 4 run.
- Increased projection_dim to 128 so the InfoNCE projector has more room.
- Reduced stem_dropout_prob to 0.08 to preserve a bit more stem detail.
- Rebalanced losses toward audio identity: lambda_relational_align=0.90, lambda_view_alignment=0.40, lambda_instance_nce=0.05, drum_view_weight=0.35.
- Cleared copied outputs so this notebook starts clean and can be rerun end-to-end.


In [ ]:
# =========================
# Config (edit this cell)
# =========================
CFG = {
    "experiment_name": "05_Resnet18_contrastive_tags_infonce",
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Data
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,

    # Optimization
    "num_epochs": 80,
    "max_steps_per_epoch": None,  # set an int here for quick probes
    "learning_rate": 8e-5,
    "weight_decay": 1e-5,
    "grad_clip_norm": 1.0,

    # Model
    "embedding_dim": 64,
    "projection_dim": 128,
    "pretrained": True,
    "stem_dropout_prob": 0.08,
    "fusion_alpha_init": 0.75,
    "drum_alpha_init": 0.20,

    # Augmentation
    "aug_enabled_train": True,
    "aug_pitch_shift_bins": 2,
    "aug_time_scale_range": (0.98, 1.02),
    "aug_gain_range": (0.98, 1.02),
    "aug_noise_std": 0.005,
    "aug_mask_prob": 0.15,
    "aug_max_mask_width": 8,
    "aug_one_second_dropout_prob": 0.03,
    "aug_one_second_width": 22,

    # Losses
    "nce_temperature": 0.12,
    "lambda_relational_align": 0.90,
    "lambda_view_alignment": 0.40,
    # Small InfoNCE term to keep song-specific audio identity without overpowering the semantic teacher.
    "lambda_instance_nce": 0.05,
    "drum_view_weight": 0.35,

    # Query + optional rerank
    "query_spotify_id": "5kgvRTKmoJChOc5PAdHZg3",  # QOTSA - Keep Your Eyes Peeled
    "rerank_tag_weight": 0.0,
    "rerank_year_weight": 0.0,
    "rerank_artist_repeat_penalty": 0.0,
    # Training logging
    "batch_logs_per_epoch": 2,

    # Quick metric evaluation (set None for full catalog)
    "metrics_max_queries": 2000,
    "metrics_sample_seed": 42,

    # Output
    "run_label": "05_Resnet18_contrastive_tags_infonce",
}

OUTPUT_DIR = (DATA_DIR / "processed" / "model_runs" / CFG["run_label"]).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("experiment:", CFG["experiment_name"])
print("device:", CFG["device"])
print("output_dir:", OUTPUT_DIR)


In [ ]:
def set_seed(seed: int) -> None:
    """Set random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def mean_or_nan(values) -> float:
    return float(np.mean(values)) if values else np.nan


set_seed(CFG["seed"])
device = torch.device(CFG["device"])


## 3) Split Load

Load the processed train/test splits and derive `clean_tags` once at load time.

- `train_df` drives optimization.
- `test_df` is used only for retrieval and quick checks.
- `clean_tags` comes from the processed `tag_list` plus notebook 2's valid tag vocabulary.

The train/test split itself is the leakage boundary. This notebook keeps that boundary intact by training only on `train_df` and using `test_df` only after training for held-out embedding build and evaluation.


In [ ]:
valid_tags = set(json.loads(VALID_TAGS.read_text()))
# Split loading stays simple: read the processed frame, then derive clean tags once.


def load_split_df(split_name: str) -> pd.DataFrame:
    frame = pd.read_parquet(DATA_DIR / "processed" / f"{split_name}.parquet").copy()
    frame["clean_tags"] = frame["tag_list"].apply(lambda tags: clean_tags(tags, valid_tags))
    return frame


train_df = load_split_df("train")
test_df = load_split_df("test")

train_indexer = TrackIndexer(train_df)
test_indexer = TrackIndexer(test_df)
# These checks protect the assumptions used later by the stem branches and the holdout evaluation.
split_overlap = len(set(train_df["spotify_id"]) & set(test_df["spotify_id"]))
assert split_overlap == 0, "train/test splits must be disjoint"
expected_stem_order = ['bass', 'drums', 'other', 'vocals']
assert train_indexer.stem_list == expected_stem_order, f"unexpected train stem order: {train_indexer.stem_list}"
assert test_indexer.stem_list == expected_stem_order, f"unexpected test stem order: {test_indexer.stem_list}"

print("train songs:", len(train_df))
print("test songs:", len(test_df))
print("train/test overlap:", split_overlap)
print("stem order:", ["full mix"] + train_indexer.stem_list)

train_df[["spotify_id", "clean_tags"]].head(3)


## 4) Semantic Teacher from Notebook 2

This section is a thin adapter from notebook 2 to notebook 4.

For each song we:
1. keep tags that exist in notebook 2,
2. remove duplicates,
3. look up the frozen tag vectors,
4. store that set of tag vectors for the loss.

$$E_i = \{e(g) : g \in G_i\}$$

Here $G_i$ is the song's clean tag set and $e(g)$ is the frozen notebook 2 embedding for tag $g$.

We also build tag sets for the test split, but that is not a training leak: optimization uses only the train-side semantic dictionaries. The test split's tag sets are created only for held-out coverage and tag-overlap diagnostics in the same frozen teacher space.


In [ ]:
tag_keys = json.loads(TAG_KEYS.read_text())
# Notebook 2 already defines the teacher space, so here we only load it and keep it fixed.
tag_vectors = torch.as_tensor(np.load(TAG_VECTORS).astype(np.float32), device=device)
tag_vectors = F.normalize(tag_vectors, dim=1, eps=1e-8)
tag_vector_map = dict(zip(tag_keys, tag_vectors))
empty_tag_matrix = torch.empty((0, tag_vectors.shape[1]), device=device)


def keep_embedded_tags(tags: list[str]) -> list[str]:
    # Keep tag order, but drop duplicates so one repeated tag does not get extra weight.
    return [tag for tag in dict.fromkeys(tags) if tag in tag_vector_map]


def build_semantic_tag_matrix(tags: list[str]) -> tuple[torch.Tensor, bool]:
    tags = keep_embedded_tags(tags)
    if not tags:
        return empty_tag_matrix.clone(), False
    return torch.stack([tag_vector_map[tag] for tag in tags], dim=0), True


def prepare_semantic_tag_sets(frame: pd.DataFrame):
    frame = frame.copy()
    # From here on, clean_tags means the notebook-2 subset that can actually enter the semantic teacher.
    frame["clean_tags"] = frame["clean_tags"].apply(keep_embedded_tags)

    track_id_to_tag_matrix = {}
    track_id_has_semantic_teacher = {}
    # Build one frozen tag-set teacher per song; section 7 compares songs set-to-set.
    for track_id, tags in frame[["spotify_id", "clean_tags"]].itertuples(index=False):
        tag_matrix, has_teacher = build_semantic_tag_matrix(list(tags))
        track_id_to_tag_matrix[track_id] = tag_matrix
        track_id_has_semantic_teacher[track_id] = has_teacher

    frame["has_semantic_teacher"] = frame["spotify_id"].map(track_id_has_semantic_teacher)
    return frame, track_id_to_tag_matrix, track_id_has_semantic_teacher


train_df, train_track_id_to_tag_matrix, train_track_id_has_semantic_teacher = prepare_semantic_tag_sets(train_df)
test_df, _, _ = prepare_semantic_tag_sets(test_df)

print("semantic teacher dim:", tag_vectors.shape[1])
print("train tracks with semantic teacher:", int(train_df["has_semantic_teacher"].sum()), "/", len(train_df))
print("test tracks with semantic teacher:", int(test_df["has_semantic_teacher"].sum()), "/", len(test_df))

train_df[["spotify_id", "clean_tags", "has_semantic_teacher"]].head(3)


## 5) Dataset and Augmentation

Section 5 only does two things:
- load the full-mix and stem spectrograms for each song,
- apply light training augmentation while keeping mix and stems aligned.


In [ ]:
class SongAugmentation:
    def __init__(
        self,
        pitch_shift_bins=2,
        time_scale_range=(0.98, 1.02),
        gain_range=(0.98, 1.02),
        noise_std=0.005,
        mask_prob=0.15,
        max_mask_width=8,
        one_second_dropout_prob=0.03,
        one_second_width=22,
        enabled=True,
    ):
        self.pitch_shift_bins = pitch_shift_bins
        self.time_scale_range = time_scale_range
        self.gain_range = gain_range
        self.noise_std = noise_std
        self.mask_prob = mask_prob
        self.max_mask_width = max_mask_width
        self.one_second_dropout_prob = one_second_dropout_prob
        self.one_second_width = one_second_width
        self.enabled = enabled

    def _time_scale(self, x, scale):
        _, h, w = x.shape
        new_w = max(8, int(round(w * scale)))
        x_scaled = F.interpolate(x.unsqueeze(0), size=(h, new_w), mode="bilinear", align_corners=False)
        x_scaled = F.interpolate(x_scaled, size=(h, w), mode="bilinear", align_corners=False)
        return x_scaled.squeeze(0)

    def _pitch_shift(self, x, shift):
        if shift == 0:
            return x
        shifted = torch.zeros_like(x)
        if shift > 0:
            shifted[:, shift:, :] = x[:, :-shift, :]
        else:
            shifted[:, :shift, :] = x[:, -shift:, :]
        return shifted

    def _time_mask(self, x):
        if random.random() > self.mask_prob:
            return x
        _, _, w = x.shape
        mask_width = random.randint(1, self.max_mask_width)
        start = random.randint(0, max(0, w - mask_width))
        x = x.clone()
        x[:, :, start:start + mask_width] = 0.0
        return x

    def _long_time_dropout(self, x):
        if random.random() > self.one_second_dropout_prob:
            return x
        _, _, w = x.shape
        mask_width = min(self.one_second_width, w)
        start = random.randint(0, max(0, w - mask_width))
        x = x.clone()
        x[:, :, start:start + mask_width] = 0.0
        return x

    def __call__(self, mix, stems):
        if not self.enabled:
            return mix, stems

        shift = random.randint(-self.pitch_shift_bins, self.pitch_shift_bins)
        scale = random.uniform(*self.time_scale_range)
        gain = random.uniform(*self.gain_range)

        # Share the coarse transform across views, then add light per-view perturbations.
        def apply(x):
            x = self._pitch_shift(x, shift)
            x = self._time_scale(x, scale)
            x = x * gain
            if self.noise_std > 0:
                x = x + torch.randn_like(x) * self.noise_std
            x = self._time_mask(x)
            x = self._long_time_dropout(x)
            return x.clamp(0.0, 1.0)

        mix = apply(mix)
        stems = torch.stack([apply(stem) for stem in stems], dim=0)
        return mix, stems


class StemSongDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.image_size = image_size
        self.transform = transform
        self.indexer = TrackIndexer(self.df)

    def __len__(self) -> int:
        return len(self.df)

    def _load_spec(self, path) -> torch.Tensor:
        array = load_png_resized(str(path), image_size=self.image_size)
        return torch.as_tensor(array, dtype=torch.float32).unsqueeze(0)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        track_id = row["spotify_id"]
        # TrackIndexer returns the full mix first, then the stems in the checked project order.
        spec_paths = self.indexer.get_spec_png_paths(track_id)

        mix = self._load_spec(spec_paths[0])
        stems = torch.stack([self._load_spec(path) for path in spec_paths[1:]], dim=0)

        if self.transform is not None:
            mix, stems = self.transform(mix, stems)

        return {"track_id": track_id, "mix": mix, "stems": stems}


In [ ]:
train_augmentation = SongAugmentation(
    pitch_shift_bins=CFG["aug_pitch_shift_bins"],
    time_scale_range=CFG["aug_time_scale_range"],
    gain_range=CFG["aug_gain_range"],
    noise_std=CFG["aug_noise_std"],
    mask_prob=CFG["aug_mask_prob"],
    max_mask_width=CFG["aug_max_mask_width"],
    one_second_dropout_prob=CFG["aug_one_second_dropout_prob"],
    one_second_width=CFG["aug_one_second_width"],
    enabled=CFG["aug_enabled_train"],
)

eval_augmentation = SongAugmentation(enabled=False)

train_dataset = StemSongDataset(train_df, image_size=CFG["image_size"], transform=train_augmentation)
eval_dataset = StemSongDataset(test_df, image_size=CFG["image_size"], transform=eval_augmentation)
assert len(train_dataset) > 0, "train split must contain at least one song"
assert len(eval_dataset) > 0, "test split must contain at least one song"

pin_memory = device.type == "cuda"
# Drop the short final train batch when possible so relational batches stay more comparable.
drop_last_train = len(train_dataset) >= CFG["batch_size"]

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    drop_last=drop_last_train,
    pin_memory=pin_memory,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=pin_memory,
)

# One eval batch is enough to confirm the tensor contract before training starts.
batch = next(iter(eval_loader))
print("train size:", len(train_dataset), "| test size:", len(eval_dataset))
print("mix shape:", tuple(batch["mix"].shape))
print("stems shape:", tuple(batch["stems"].shape))


## 6) Model: Stem Late-Fusion ResNet18

The model stays simple:
1. one shared ResNet18 encodes the mix and every stem,
2. bass/other/vocals are pooled into a harmonic branch,
3. drums stay separate,
4. the final song embedding is a gated mix of mix, harmonic, and drum embeddings.

Final fusion:
$$
z = \operatorname{normalize}(m + \alpha_h h + \alpha_d d)
$$

$z$ is the retrieval embedding used everywhere downstream. The projection head supports a small same-song InfoNCE stabilizer between mix and harmonic views. In this notebook it is active with a small weight, so the embedding keeps a bit more song-specific audio identity while staying mostly driven by the semantic teacher.


In [ ]:
class ProjectionHead(nn.Module):
    """Small MLP used only for the instance InfoNCE term."""
    def __init__(self, input_dim: int, projection_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(inplace=True),
            nn.Linear(input_dim, projection_dim),
        )

    def forward(self, x):
        return self.net(x)


class StemLateFusionResNet18(nn.Module):
    def __init__(
        self,
        embedding_dim=128,
        projection_dim=64,
        pretrained=True,
        imagenet_input_norm=None,
        fusion_alpha_init=0.7,
        drum_alpha_init=0.3,
        num_stems=4,
        stem_dropout_prob=0.1,
        harmonic_indices=(0, 2, 3),
        drum_index=1,
    ):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Linear(in_features, embedding_dim)

        self.encoder = backbone
        self.projection_head = ProjectionHead(embedding_dim, projection_dim)
        if imagenet_input_norm is None:
            imagenet_input_norm = pretrained
        self.imagenet_input_norm = bool(imagenet_input_norm)
        self.register_buffer(
            "imagenet_mean",
            torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1),
            persistent=False,
        )
        self.register_buffer(
            "imagenet_std",
            torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1),
            persistent=False,
        )

        self.fusion_alpha_logit = nn.Parameter(torch.tensor(fusion_alpha_init, dtype=torch.float32).logit())
        self.drum_alpha_logit = nn.Parameter(torch.tensor(drum_alpha_init, dtype=torch.float32).logit())

        self.num_stems = int(num_stems)
        self.harmonic_indices = tuple(int(idx) for idx in harmonic_indices)
        self.drum_index = int(drum_index)
        self.stem_logits = nn.Parameter(torch.zeros(self.num_stems, dtype=torch.float32))
        self.harmonic_logits = nn.Parameter(torch.zeros(len(self.harmonic_indices), dtype=torch.float32))
        self.register_buffer(
            "harmonic_index_tensor",
            torch.tensor(self.harmonic_indices, dtype=torch.long),
            persistent=False,
        )
        self.stem_dropout_prob = float(stem_dropout_prob)

    def _encode_inputs(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        if self.imagenet_input_norm:
            x = (x - self.imagenet_mean) / self.imagenet_std
        return self.encoder(x)

    def forward(self, mix, stems):
        batch_size, num_stems, channels, height, width = stems.shape

        mix_embedding = self._encode_inputs(mix)
        stem_inputs = stems.reshape(batch_size * num_stems, channels, height, width)
        stem_embeddings = self._encode_inputs(stem_inputs).reshape(batch_size, num_stems, -1)
        stem_embeddings = F.normalize(stem_embeddings, dim=2, eps=1e-8)

        # stem_logits learn a global prior over stems across the whole dataset.
        stem_weights = torch.softmax(self.stem_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
        if self.training and self.stem_dropout_prob > 0:
            keep_mask = (torch.rand(batch_size, num_stems, device=stems.device) > self.stem_dropout_prob).float()
            keep_mask = torch.where(keep_mask.sum(dim=1, keepdim=True) == 0, torch.ones_like(keep_mask), keep_mask)
            stem_weights = stem_weights * keep_mask
        stem_weights = stem_weights / stem_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)

        harmonic_stems = stem_embeddings[:, self.harmonic_index_tensor, :]
        # harmonic_logits then reweight only the harmonic subset inside that global stem prior.
        harmonic_weights = torch.softmax(self.harmonic_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
        harmonic_weights = harmonic_weights * stem_weights[:, self.harmonic_index_tensor]
        harmonic_weights = torch.where(
            harmonic_weights.sum(dim=1, keepdim=True) == 0,
            torch.ones_like(harmonic_weights),
            harmonic_weights,
        )
        harmonic_weights = harmonic_weights / harmonic_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        harmonic_embedding = torch.sum(harmonic_stems * harmonic_weights.unsqueeze(-1), dim=1)
        drum_embedding = stem_embeddings[:, self.drum_index, :] * stem_weights[:, self.drum_index].unsqueeze(-1)

        # Normalize branches before fusion so the learned gates control contribution size.
        mix_embedding = F.normalize(mix_embedding, dim=1, eps=1e-8)
        harmonic_embedding = F.normalize(harmonic_embedding, dim=1, eps=1e-8)
        drum_embedding = F.normalize(drum_embedding, dim=1, eps=1e-8)

        alpha_h = torch.sigmoid(self.fusion_alpha_logit)
        alpha_d = torch.sigmoid(self.drum_alpha_logit)
        song_embedding = F.normalize(
            mix_embedding + alpha_h * harmonic_embedding + alpha_d * drum_embedding,
            dim=1,
            eps=1e-8,
        )

        harmonic_projection = self.projection_head(harmonic_embedding)

        return {
            "mix_embedding": mix_embedding,
            "stem_embeddings": stem_embeddings,
            "fused_stem_embedding": harmonic_embedding,
            "harmonic_embedding": harmonic_embedding,
            "drum_embedding": drum_embedding,
            "stem_weights": stem_weights,
            "harmonic_weights": harmonic_weights,
            "song_embedding": song_embedding,
            "mix_projection": self.projection_head(mix_embedding),
            "harmonic_projection": harmonic_projection,
            "stem_projection": harmonic_projection,
        }


## 7) Losses

The notebook now uses three training terms, with the InfoNCE term kept intentionally small.

- $L_{\mathrm{rel}}$: make audio-song similarity follow notebook 2 tag-set similarity.
- $L_{\mathrm{view}}$: keep the fused embedding close to the song's own audio views.
- $L_{\mathrm{inst}}$: small same-song InfoNCE between mix and harmonic views.

In the current config, $\lambda_{\mathrm{inst}} = 0.05$, so $L_{\mathrm{inst}}$ is active but modest. The goal is to preserve some cross-view audio identity without letting the contrastive term overpower the tag-guided structure.


## Overall Training Loss
 
 $$
 L = \lambda_{\mathrm{rel}} L_{\mathrm{rel}} + \lambda_{\mathrm{view}} L_{\mathrm{view}} + \lambda_{\mathrm{inst}} L_{\mathrm{inst}}
 $$
 
 with
 $$
 L_{\mathrm{rel}} = \operatorname*{mean}_{i<j} \left(\cos(z_i, z_j) - S_{\mathrm{tag}}(E_i, E_j)\right)^2
 $$
 $$
 L_{\mathrm{view}} = \tfrac{1}{2}\left((1 - \cos(z, m)) + (1 - \cos(z, h))\right) + w_d (1 - \cos(z, d))
 $$
 $$
 L_{\mathrm{inst}} = \text{symmetric InfoNCE between mix and harmonic projections}
 $$
 
 Here $S_{\mathrm{tag}}(E_i, E_j)$ is the symmetric best-match similarity between the two tag sets.
 
 In practice, the current notebook trains with all three terms. The semantic relational loss still provides the backbone, the view loss keeps the fused embedding tied to the audio branches, and the small nonzero $\lambda_{\mathrm{inst}}$ adds extra same-song audio consistency.


In [ ]:
def symmetric_info_nce_loss(query_projection, key_projection, temperature=0.1):
    """Bidirectional InfoNCE used only for same-song multi-view alignment."""
    query_projection = F.normalize(query_projection, dim=1, eps=1e-8)
    key_projection = F.normalize(key_projection, dim=1, eps=1e-8)

    logits = (query_projection @ key_projection.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_query_to_key = F.cross_entropy(logits, labels)
    loss_key_to_query = F.cross_entropy(logits.T, labels)
    return 0.5 * (loss_query_to_key + loss_key_to_query)


def gather_semantic_tag_sets(track_ids, track_id_to_tag_matrix, track_id_has_semantic_teacher):
    semantic_tag_sets = [track_id_to_tag_matrix[track_id] for track_id in track_ids]
    # Some songs may have no valid notebook 2 tags; the mask keeps them out of the relational term.
    mask_device = semantic_tag_sets[0].device if semantic_tag_sets else torch.device("cpu")
    semantic_mask = torch.tensor(
        [track_id_has_semantic_teacher[track_id] for track_id in track_ids],
        device=mask_device,
        dtype=torch.bool,
    )
    return semantic_tag_sets, semantic_mask


def symmetric_best_match_similarity(tag_matrix_a, tag_matrix_b):
    # Each tag looks for its best match in the other song, then we average both directions.
    pairwise_sim = F.normalize(tag_matrix_a, dim=1, eps=1e-8) @ F.normalize(tag_matrix_b, dim=1, eps=1e-8).T
    a_to_b = pairwise_sim.max(dim=1).values.mean()
    b_to_a = pairwise_sim.max(dim=0).values.mean()
    return 0.5 * (a_to_b + b_to_a)


def relational_semantic_loss(song_embeddings, semantic_tag_sets, semantic_mask):
    """Match batchwise audio similarity to notebook 2 tag-set similarity."""
    valid_indices = semantic_mask.nonzero(as_tuple=False).flatten().tolist()
    valid_count = len(valid_indices)
    # If fewer than two songs in the batch have semantic teachers, the relational term is inactive.
    if valid_count < 2:
        zero = song_embeddings.new_tensor(0.0)
        return zero, song_embeddings.new_tensor(float("nan")), valid_count

    song_valid = F.normalize(song_embeddings[semantic_mask], dim=1, eps=1e-8)
    tag_sets_valid = [semantic_tag_sets[idx] for idx in valid_indices]

    audio_pair_sims = []
    tag_pair_sims = []
    for left in range(valid_count - 1):
        for right in range(left + 1, valid_count):
            audio_pair_sims.append(torch.dot(song_valid[left], song_valid[right]))
            tag_pair_sims.append(symmetric_best_match_similarity(tag_sets_valid[left], tag_sets_valid[right]))

    audio_pair_sims = torch.stack(audio_pair_sims)
    tag_pair_sims = torch.stack(tag_pair_sims)
    loss = F.mse_loss(audio_pair_sims, tag_pair_sims)
    mean_gap = (audio_pair_sims - tag_pair_sims).abs().mean()
    return loss, mean_gap, valid_count


## 8) Training Loop

For each batch we:
1. run the model on mix and stems,
2. gather the frozen semantic tag sets for those songs,
3. compute the relational loss plus the audio-side view regularizer,
4. add the small InfoNCE term,
5. update the model with Adam.

The history table keeps the training summary easy to read: total loss, the three loss terms, learning rate, fusion gates, relational gap, and semantic teacher coverage.


In [ ]:
def train_one_run(cfg, train_loader, train_track_id_to_tag_matrix, train_track_id_has_semantic_teacher, device):
    """Train one experiment run from CFG and return model + epoch history."""
    model = StemLateFusionResNet18(
        embedding_dim=cfg["embedding_dim"],
        projection_dim=cfg["projection_dim"],
        pretrained=cfg["pretrained"],
        imagenet_input_norm=cfg["pretrained"],
        fusion_alpha_init=cfg["fusion_alpha_init"],
        drum_alpha_init=cfg["drum_alpha_init"],
        stem_dropout_prob=cfg["stem_dropout_prob"],
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg["num_epochs"]))

    history = []
    for epoch in range(cfg["num_epochs"]):
        model.train()
        epoch_losses = []
        epoch_rel_losses = []
        epoch_view_losses = []
        epoch_inst_losses = []
        epoch_relational_gaps = []
        epoch_semantic_coverages = []

        for step, batch in enumerate(train_loader):
            if cfg["max_steps_per_epoch"] is not None and step >= cfg["max_steps_per_epoch"]:
                break

            # Forward pass: one fused retrieval embedding plus the audio branches used by the auxiliary losses.
            mix = batch["mix"].to(device)
            stems = batch["stems"].to(device)
            outputs = model(mix, stems)

            # Pull the frozen semantic teachers for just this batch of songs.
            semantic_tag_sets, semantic_mask = gather_semantic_tag_sets(
                batch["track_id"],
                track_id_to_tag_matrix=train_track_id_to_tag_matrix,
                track_id_has_semantic_teacher=train_track_id_has_semantic_teacher,
            )
            relational_loss, relational_gap, semantic_count = relational_semantic_loss(
                outputs["song_embedding"],
                semantic_tag_sets,
                semantic_mask,
            )

            mix_alignment = 1.0 - F.cosine_similarity(
                outputs["song_embedding"],
                outputs["mix_embedding"],
                dim=1,
                eps=1e-8,
            ).mean()
            harmonic_alignment = 1.0 - F.cosine_similarity(
                outputs["song_embedding"],
                outputs["harmonic_embedding"],
                dim=1,
                eps=1e-8,
            ).mean()
            drum_alignment = 1.0 - F.cosine_similarity(
                outputs["song_embedding"],
                outputs["drum_embedding"],
                dim=1,
                eps=1e-8,
            ).mean()
            # Keep the fused retrieval embedding tied to the audio evidence that produced it.
            view_alignment_loss = (
                0.5 * (mix_alignment + harmonic_alignment)
                + cfg["drum_view_weight"] * drum_alignment
            )

            # Apply the small InfoNCE term when the config enables extra cross-view audio consistency.
            if cfg["lambda_instance_nce"] > 0:
                instance_nce_loss = symmetric_info_nce_loss(
                    outputs["mix_projection"],
                    outputs["harmonic_projection"],
                    temperature=cfg["nce_temperature"],
                )
            else:
                instance_nce_loss = outputs["song_embedding"].new_tensor(0.0)

            loss = cfg["lambda_relational_align"] * relational_loss
            loss = loss + cfg["lambda_view_alignment"] * view_alignment_loss
            loss = loss + cfg["lambda_instance_nce"] * instance_nce_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["grad_clip_norm"])
            optimizer.step()

            epoch_losses.append(float(loss.detach().cpu()))
            epoch_rel_losses.append(float(relational_loss.detach().cpu()))
            epoch_view_losses.append(float(view_alignment_loss.detach().cpu()))
            epoch_inst_losses.append(float(instance_nce_loss.detach().cpu()))
            if semantic_count > 1:
                epoch_relational_gaps.append(float(relational_gap.detach().cpu()))
            epoch_semantic_coverages.append(float(semantic_count / max(len(batch["track_id"]), 1)))
            steps_this_epoch = cfg["max_steps_per_epoch"] or len(train_loader)
            log_every = max(1, int(np.ceil(steps_this_epoch / max(int(cfg.get("batch_logs_per_epoch", 0)), 1))))
            should_log_batch = ((step + 1) % log_every == 0) or ((step + 1) == steps_this_epoch)
            if cfg.get("batch_logs_per_epoch", 0) > 0 and should_log_batch:
                print(
                    f"epoch {epoch + 1:02d}/{cfg['num_epochs']} | "
                    f"batch {step + 1:03d}/{steps_this_epoch:03d} | loss={epoch_losses[-1]:.4f} | "
                    f"rel={epoch_rel_losses[-1]:.4f} | view={epoch_view_losses[-1]:.4f} | "
                    f"inst={epoch_inst_losses[-1]:.4f} | sem_cov={epoch_semantic_coverages[-1]:.2%}"
                )

        # Store a compact epoch summary so the notebook stays easy to scan after training.
        epoch_summary = {
            "epoch": epoch + 1,
            "avg_loss": mean_or_nan(epoch_losses),
            "avg_rel_loss": mean_or_nan(epoch_rel_losses),
            "avg_view_loss": mean_or_nan(epoch_view_losses),
            "avg_inst_loss": mean_or_nan(epoch_inst_losses),
            "lr": optimizer.param_groups[0]["lr"],
            "harmonic_alpha": float(torch.sigmoid(model.fusion_alpha_logit).detach().cpu()),
            "drum_alpha": float(torch.sigmoid(model.drum_alpha_logit).detach().cpu()),
            "relational_gap": mean_or_nan(epoch_relational_gaps),
            "semantic_teacher_coverage": mean_or_nan(epoch_semantic_coverages),
        }
        history.append(epoch_summary)

        print(
            f"epoch {epoch_summary['epoch']:02d}/{cfg['num_epochs']} | "
            f"loss={epoch_summary['avg_loss']:.4f} | rel={epoch_summary['avg_rel_loss']:.4f} | "
            f"view={epoch_summary['avg_view_loss']:.4f} | inst={epoch_summary['avg_inst_loss']:.4f} | "
            f"lr={epoch_summary['lr']:.2e} | alpha_h={epoch_summary['harmonic_alpha']:.3f} | "
            f"alpha_d={epoch_summary['drum_alpha']:.3f} | rel_gap={epoch_summary['relational_gap']:.4f} | "
            f"sem_cov={epoch_summary['semantic_teacher_coverage']:.2%}"
        )
        scheduler.step()

    return model, pd.DataFrame(history)


In [ ]:
model, train_history = train_one_run(
    cfg=CFG,
    train_loader=train_loader,
    train_track_id_to_tag_matrix=train_track_id_to_tag_matrix,
    train_track_id_has_semantic_teacher=train_track_id_has_semantic_teacher,
    device=device,
)

train_history


In [ ]:
plt.figure(figsize=(8, 4))
for column, label in [
    ("avg_loss", "total"),
    ("avg_rel_loss", "rel"),
    ("avg_view_loss", "view"),
    ("avg_inst_loss", "inst"),
]:
    plt.plot(train_history["epoch"], train_history[column], marker="o", label=label)

plt.title("Training Curves")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 9) Embedding Build and Retrieval

After training, encode the test split once and retrieve neighbors by cosine similarity on `song_embedding`.

That keeps evaluation easy to read:
- train on the train split,
- freeze the model,
- rank held-out test songs by cosine similarity.


In [ ]:
def parse_tag_list(value) -> list[str]:
    # Use one tag parser for reranking, examples, and quick metrics so these cells stay consistent.
    if isinstance(value, (list, tuple, set)):
        return [str(v).strip() for v in value if str(v).strip()]
    if isinstance(value, np.ndarray):
        return [str(v).strip() for v in value.tolist() if str(v).strip()]
    if isinstance(value, str):
        return [v.strip() for v in value.split(",") if v.strip()]
    return []


def parse_tag_set(value) -> set[str]:
    return set(parse_tag_list(value))


def tag_jaccard(a_tags: set[str], b_tags: set[str]) -> float:
    union = a_tags | b_tags
    return len(a_tags & b_tags) / len(union) if union else 0.0


@torch.no_grad()
def build_song_embeddings(model, loader, device):
    # Encode the full test split once, then reuse the cached matrix for retrieval and metrics.
    model.eval()
    track_ids = []
    song_embeddings = []
    mix_embeddings = []
    # Keep the old artifact name, but this is the harmonic branch used in the late fusion.
    stem_embeddings = []

    for batch in loader:
        mix = batch["mix"].to(device)
        stems = batch["stems"].to(device)
        out = model(mix, stems)

        track_ids.extend(batch["track_id"])
        song_embeddings.append(F.normalize(out["song_embedding"], dim=1, eps=1e-8).cpu())
        mix_embeddings.append(F.normalize(out["mix_embedding"], dim=1, eps=1e-8).cpu())
        stem_embeddings.append(F.normalize(out["fused_stem_embedding"], dim=1, eps=1e-8).cpu())

    return {
        "track_ids": track_ids,
        "song_embeddings": torch.cat(song_embeddings, dim=0).numpy(),
        "mix_embeddings": torch.cat(mix_embeddings, dim=0).numpy(),
        "stem_embeddings": torch.cat(stem_embeddings, dim=0).numpy(),
    }


def topk_cosine(embeddings, query_vec, k=10, exclude_idx=None):
    emb = torch.as_tensor(embeddings, dtype=torch.float32)
    q = torch.as_tensor(query_vec, dtype=torch.float32)
    sims = emb @ q

    if exclude_idx is not None:
        sims[exclude_idx] = float("-inf")

    max_k = emb.shape[0] - (1 if exclude_idx is not None else 0)
    top_vals, top_idx = torch.topk(sims, k=min(int(k), int(max_k)), largest=True, sorted=True)
    return top_idx.cpu().numpy(), top_vals.cpu().numpy()


def recommend_from_query(embedding_df, emb, query_spotify_id, k=10):
    query_idx = int(embedding_df.index[embedding_df["spotify_id"] == query_spotify_id][0])
    idx, scores = topk_cosine(emb, emb[query_idx], k=k, exclude_idx=query_idx)
    recs = embedding_df.iloc[idx].copy()
    recs["similarity"] = scores
    return query_idx, emb[query_idx], recs


def recommend_from_query_reranked(embedding_df, emb, query_spotify_id, k=10, candidate_pool=100):
    candidate_pool = max(int(candidate_pool), int(k))
    query_idx, _, recs = recommend_from_query(embedding_df, emb, query_spotify_id, k=candidate_pool)
    qrow = embedding_df.iloc[query_idx]
    qtags = parse_tag_set(qrow.get("clean_tags", []))

    recs = recs.copy()
    recs["tag_jaccard"] = recs["clean_tags"].apply(lambda t: tag_jaccard(qtags, parse_tag_set(t)))
    recs["year_bonus"] = -recs["year"].apply(
        lambda y: 0.0 if pd.isna(y) or pd.isna(qrow.get("year", np.nan)) else abs(float(y) - float(qrow["year"]))
    )
    recs["artist_penalty"] = (recs["artist"] == qrow["artist"]).astype(float)

    recs["rerank_score"] = (
        recs["similarity"]
        + CFG["rerank_tag_weight"] * recs["tag_jaccard"]
        + CFG["rerank_year_weight"] * recs["year_bonus"]
        - CFG["rerank_artist_repeat_penalty"] * recs["artist_penalty"]
    )
    recs = recs.sort_values("rerank_score", ascending=False).head(k)
    return query_idx, emb[query_idx], recs


def reranking_enabled(cfg):
    return any(
        float(cfg[key]) != 0.0
        for key in ["rerank_tag_weight", "rerank_year_weight", "rerank_artist_repeat_penalty"]
    )


In [ ]:
embedding_out = build_song_embeddings(model, eval_loader, device)
track_ids = embedding_out["track_ids"]
song_embeddings = embedding_out["song_embeddings"]
mix_embeddings = embedding_out["mix_embeddings"]
stem_embeddings = embedding_out["stem_embeddings"]

embedding_df = test_df[["spotify_id", "artist", "name", "tags", "clean_tags", "year", "has_semantic_teacher"]].copy()
embedding_df = embedding_df.set_index("spotify_id").loc[track_ids].reset_index()

query_id = CFG["query_spotify_id"]
if query_id not in set(embedding_df["spotify_id"]):
    query_id = embedding_df.iloc[0]["spotify_id"]
    print("CFG query id not found in test split; using first test track instead:", query_id)

_, _, recs_base = recommend_from_query(embedding_df, song_embeddings, query_id, k=10)
recs_rerank = None
if reranking_enabled(CFG):
    _, _, recs_rerank = recommend_from_query_reranked(embedding_df, song_embeddings, query_id, k=10, candidate_pool=100)

query_row = embedding_df.loc[embedding_df["spotify_id"] == query_id].iloc[0]
print("query:", query_row["artist"], "-", query_row["name"])
print("query clean tags:", query_row["clean_tags"])
print()
print("Baseline top-10")
display(recs_base[["artist", "name", "spotify_id", "similarity"]])

if recs_rerank is not None:
    print()
    print("Optional reranked top-10")
    display(recs_rerank[["artist", "name", "spotify_id", "similarity", "rerank_score"]])
else:
    print()
    print("Optional reranking is disabled because all rerank weights are 0.0.")


## 10) Lightweight Quality Checks

These are quick held-out checks:
- `artist_hit@10`
- `tag_overlap_hit@10`
- `semantic_teacher_coverage`

They are meant to be fast sanity checks, not the final word on recommendation quality.


In [ ]:
def sample_eval_df(embedding_df, max_queries=None, seed=42):
    # Quick metrics can subsample queries so this section stays fast on larger catalogs.
    if len(embedding_df) == 0 or max_queries is None or max_queries >= len(embedding_df):
        return embedding_df.reset_index(drop=True)
    if max_queries <= 0:
        return embedding_df.iloc[0:0].copy()
    return embedding_df.sample(max_queries, random_state=seed).reset_index(drop=True)


def queries_with_clean_tags(eval_df):
    return eval_df[eval_df["clean_tags"].apply(lambda tags: len(parse_tag_set(tags)) > 0)].reset_index(drop=True)


def artist_hit_at_k(embedding_df, emb, k=10, max_queries=None, seed=42):
    eval_df = sample_eval_df(embedding_df, max_queries=max_queries, seed=seed)
    if len(eval_df) == 0:
        return float("nan"), 0

    hits = []
    for _, row in eval_df.iterrows():
        _, _, recs = recommend_from_query(embedding_df, emb, row["spotify_id"], k=k)
        hits.append(float((recs["artist"] == row["artist"]).any()))
    return float(np.mean(hits)), len(eval_df)


def tag_overlap_hit_at_k(embedding_df, emb, k=10, max_queries=None, seed=42):
    eval_df = queries_with_clean_tags(sample_eval_df(embedding_df, max_queries=max_queries, seed=seed))
    if len(eval_df) == 0:
        return float("nan"), 0

    hits = []
    for _, row in eval_df.iterrows():
        qtags = parse_tag_set(row.get("clean_tags", []))
        _, _, recs = recommend_from_query(embedding_df, emb, row["spotify_id"], k=k)
        overlap = any(len(qtags & parse_tag_set(tags)) > 0 for tags in recs["clean_tags"].tolist())
        hits.append(float(overlap))
    return float(np.mean(hits)), len(eval_df)


def semantic_teacher_coverage(embedding_df):
    # Coverage answers: how much of the test split actually has a notebook 2 teacher set?
    return float(embedding_df["has_semantic_teacher"].mean())


artist_hit_10, artist_query_count = artist_hit_at_k(
    embedding_df,
    song_embeddings,
    k=10,
    max_queries=CFG["metrics_max_queries"],
    seed=CFG["metrics_sample_seed"],
)
tag_overlap_hit_10, tag_overlap_query_count = tag_overlap_hit_at_k(
    embedding_df,
    song_embeddings,
    k=10,
    max_queries=CFG["metrics_max_queries"],
    seed=CFG["metrics_sample_seed"],
)

metrics = pd.DataFrame(
    {
        "metric": ["artist_hit@10", "tag_overlap_hit@10", "semantic_teacher_coverage"],
        "value": [
            artist_hit_10,
            tag_overlap_hit_10,
            semantic_teacher_coverage(embedding_df),
        ],
        "queries": [artist_query_count, tag_overlap_query_count, len(embedding_df)],
    }
)
metrics

print(f"artist_hit@10 queries evaluated: {artist_query_count}")
print(f"tag_overlap_hit@10 queries evaluated: {tag_overlap_query_count}")


## Appendix: Optional Metadata Reranking

This stays outside the main retrieval path.

If you later enable nonzero rerank weights, the notebook can rescore a cosine candidate pool with three metadata terms:
- $J_{\mathrm{tag}}$: clean-tag overlap,
- $b_{\mathrm{year}}$: closer release years score higher,
- $p_{\mathrm{artist}}$: same-artist penalty.

Rerank score:
$$
s_{\mathrm{rerank}} = s_{\mathrm{cos}} + w_t J_{\mathrm{tag}} + w_y b_{\mathrm{year}} - w_a p_{\mathrm{artist}}
$$

By default, $w_t = w_y = w_a = 0$, so the notebook story stays focused on baseline cosine retrieval from the learned embedding.


## Run Summary

- `CFG` is the single control panel for the run.
- Training uses `data/processed/train.parquet`.
- Retrieval and quick checks use `data/processed/test.parquet`.
- The learned `song_embedding` is trained so audio similarity matches notebook 2 tag-set similarity.
- Metadata reranking is optional and disabled by default.


## 11) Artifact Saving

Save the model, the test-set embeddings, and a small manifest so the run can be reloaded later without relying on anything outside this notebook.


In [ ]:
# Save artifacts for reuse.
checkpoint_path = OUTPUT_DIR / "checkpoint.pt"
embeddings_path = OUTPUT_DIR / "embeddings.npz"
manifest_path = OUTPUT_DIR / "run_manifest.json"

def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return [_json_safe(v) for v in value.tolist()]
    if isinstance(value, np.generic):
        return _json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

torch.save(
    # Save enough to rebuild the model and reuse the held-out embedding matrix later.
    {
        "model_state_dict": model.state_dict(),
        "embedding_dim": CFG["embedding_dim"],
        "projection_dim": CFG["projection_dim"],
        "model_init_kwargs": {
            "embedding_dim": CFG["embedding_dim"],
            "projection_dim": CFG["projection_dim"],
            "pretrained": CFG["pretrained"],
            "imagenet_input_norm": model.imagenet_input_norm,
            "fusion_alpha_init": CFG["fusion_alpha_init"],
            "drum_alpha_init": CFG["drum_alpha_init"],
            "num_stems": model.num_stems,
            "stem_dropout_prob": model.stem_dropout_prob,
            "harmonic_indices": list(model.harmonic_indices),
            "drum_index": model.drum_index,
        },
        "cfg": CFG,
        "train_history": train_history.to_dict(orient="records"),
    },
    checkpoint_path,
)

np.savez(
    embeddings_path,
    song_embeddings=song_embeddings.astype(np.float32),
    mix_embeddings=mix_embeddings.astype(np.float32),
    stem_embeddings=stem_embeddings.astype(np.float32),
    spotify_id=np.asarray(track_ids, dtype=str),
)

manifest = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "experiment_name": CFG["experiment_name"],
    "reference_note": "Fused song embedding trained so audio similarity matches frozen notebook 2 tag-set similarity, with light audio view regularization.",
    "run_label": CFG["run_label"],
    "checkpoint_path": str(checkpoint_path),
    "embeddings_path": str(embeddings_path),
    "test_split_path": str((DATA_DIR / "processed" / "test.parquet").resolve()),
    "num_saved_tracks": int(len(track_ids)),
    "metrics": metrics.to_dict(orient="records"),
}
manifest_path.write_text(json.dumps(_json_safe(manifest), indent=2))

print("saved checkpoint:", checkpoint_path)
print("saved embeddings:", embeddings_path)
print("saved manifest:", manifest_path)


## 12) Load Saved Artifacts Later

Use this in a fresh kernel to reload the saved model and saved test embeddings from this notebook's run directory.


In [ ]:
# Fresh-kernel reload cell: mirror the minimal model code needed for inference.
import json
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from torchvision.models import resnet18

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists() and ROOT_DIR.parent.exists():
    ROOT_DIR = ROOT_DIR.parent
SRC_DIR = ROOT_DIR / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

default_run_label = "05_Resnet18_contrastive_tags_infonce"
run_label = CFG.get("run_label", default_run_label) if "CFG" in globals() else default_run_label
data_dir = DATA_DIR if "DATA_DIR" in globals() else (ROOT_DIR / "data").resolve()

output_dir = (data_dir / "processed" / "model_runs" / run_label).resolve()
load_checkpoint_path = output_dir / "checkpoint.pt"
load_embeddings_path = output_dir / "embeddings.npz"
load_manifest_path = output_dir / "run_manifest.json"

loaded_test_df = pd.read_parquet(data_dir / "processed" / "test.parquet")
loaded_manifest = json.loads(load_manifest_path.read_text()) if load_manifest_path.exists() else None
if loaded_manifest is not None:
    print("loaded manifest:", load_manifest_path)

loaded_song_embeddings = None
loaded_mix_embeddings = None
loaded_stem_embeddings = None
loaded_track_ids = None

if load_embeddings_path.exists():
    loaded_npz = np.load(load_embeddings_path)
    loaded_song_embeddings = loaded_npz["song_embeddings"]
    loaded_mix_embeddings = loaded_npz["mix_embeddings"]
    loaded_stem_embeddings = loaded_npz["stem_embeddings"]
    loaded_track_ids = loaded_npz["spotify_id"].astype(str).tolist()
    print("loaded embeddings:", load_embeddings_path)
    print("song embeddings shape:", loaded_song_embeddings.shape)
    print("num track ids:", len(loaded_track_ids))
else:
    print("embeddings file not found:", load_embeddings_path)

if "ProjectionHead" not in globals():
    # Define the classes inline so this cell works without rerunning the training section.
    class ProjectionHead(nn.Module):
        def __init__(self, input_dim: int, projection_dim: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, input_dim),
                nn.ReLU(inplace=True),
                nn.Linear(input_dim, projection_dim),
            )

        def forward(self, x):
            return self.net(x)

if "StemLateFusionResNet18" not in globals():
    class StemLateFusionResNet18(nn.Module):
        def __init__(
            self,
            embedding_dim=64,
            projection_dim=64,
            pretrained=False,
            imagenet_input_norm=None,
            fusion_alpha_init=0.7,
            drum_alpha_init=0.3,
            num_stems=4,
            stem_dropout_prob=0.1,
            harmonic_indices=(0, 2, 3),
            drum_index=1,
        ):
            super().__init__()
            backbone = resnet18(weights=None)
            in_features = backbone.fc.in_features
            backbone.fc = nn.Linear(in_features, embedding_dim)

            self.encoder = backbone
            self.projection_head = ProjectionHead(embedding_dim, projection_dim)
            if imagenet_input_norm is None:
                imagenet_input_norm = pretrained
            self.imagenet_input_norm = bool(imagenet_input_norm)
            self.register_buffer(
                "imagenet_mean",
                torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1),
                persistent=False,
            )
            self.register_buffer(
                "imagenet_std",
                torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1),
                persistent=False,
            )

            self.fusion_alpha_logit = nn.Parameter(torch.tensor(fusion_alpha_init, dtype=torch.float32).logit())
            self.drum_alpha_logit = nn.Parameter(torch.tensor(drum_alpha_init, dtype=torch.float32).logit())
            self.num_stems = int(num_stems)
            self.harmonic_indices = tuple(int(idx) for idx in harmonic_indices)
            self.drum_index = int(drum_index)
            self.stem_logits = nn.Parameter(torch.zeros(self.num_stems, dtype=torch.float32))
            self.harmonic_logits = nn.Parameter(torch.zeros(len(self.harmonic_indices), dtype=torch.float32))
            self.register_buffer(
                "harmonic_index_tensor",
                torch.tensor(self.harmonic_indices, dtype=torch.long),
                persistent=False,
            )
            self.stem_dropout_prob = float(stem_dropout_prob)

        def _encode_inputs(self, x):
            if x.shape[1] == 1:
                x = x.repeat(1, 3, 1, 1)
            if self.imagenet_input_norm:
                x = (x - self.imagenet_mean) / self.imagenet_std
            return self.encoder(x)

        def forward(self, mix, stems):
            batch_size, num_stems, channels, height, width = stems.shape
            mix_embedding = self._encode_inputs(mix)
            stem_inputs = stems.reshape(batch_size * num_stems, channels, height, width)
            stem_embeddings = self._encode_inputs(stem_inputs).reshape(batch_size, num_stems, -1)
            stem_embeddings = F.normalize(stem_embeddings, dim=2, eps=1e-8)

            stem_weights = torch.softmax(self.stem_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
            stem_weights = stem_weights / stem_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)

            harmonic_stems = stem_embeddings[:, self.harmonic_index_tensor, :]
            harmonic_weights = torch.softmax(self.harmonic_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
            harmonic_weights = harmonic_weights * stem_weights[:, self.harmonic_index_tensor]
            harmonic_weights = torch.where(
                harmonic_weights.sum(dim=1, keepdim=True) == 0,
                torch.ones_like(harmonic_weights),
                harmonic_weights,
            )
            harmonic_weights = harmonic_weights / harmonic_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
            harmonic_embedding = torch.sum(harmonic_stems * harmonic_weights.unsqueeze(-1), dim=1)
            drum_embedding = stem_embeddings[:, self.drum_index, :] * stem_weights[:, self.drum_index].unsqueeze(-1)

            mix_embedding = F.normalize(mix_embedding, dim=1, eps=1e-8)
            harmonic_embedding = F.normalize(harmonic_embedding, dim=1, eps=1e-8)
            drum_embedding = F.normalize(drum_embedding, dim=1, eps=1e-8)

            alpha_h = torch.sigmoid(self.fusion_alpha_logit)
            alpha_d = torch.sigmoid(self.drum_alpha_logit)
            song_embedding = F.normalize(
                mix_embedding + alpha_h * harmonic_embedding + alpha_d * drum_embedding,
                dim=1,
                eps=1e-8,
            )

            harmonic_projection = self.projection_head(harmonic_embedding)
            return {
                "song_embedding": song_embedding,
                "mix_embedding": mix_embedding,
                "fused_stem_embedding": harmonic_embedding,
                "harmonic_embedding": harmonic_embedding,
                "drum_embedding": drum_embedding,
                "mix_projection": self.projection_head(mix_embedding),
                "harmonic_projection": harmonic_projection,
                "stem_projection": harmonic_projection,
            }

loaded_model = None
if load_checkpoint_path.exists():
    device = torch.device(CFG["device"] if "CFG" in globals() else ("cuda" if torch.cuda.is_available() else "cpu"))
    checkpoint = torch.load(load_checkpoint_path, map_location=device, weights_only=False)
    model_init_kwargs = checkpoint.get("model_init_kwargs", {}).copy()
    if not model_init_kwargs:
        checkpoint_cfg = checkpoint.get("cfg", {})
        model_init_kwargs = {
            "embedding_dim": int(checkpoint["embedding_dim"]),
            "projection_dim": int(checkpoint["projection_dim"]),
            "pretrained": False,
            "imagenet_input_norm": bool(checkpoint_cfg.get("pretrained", False)),
        }
    else:
        model_init_kwargs["pretrained"] = False
    loaded_model = StemLateFusionResNet18(**model_init_kwargs).to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    print("loaded model:", load_checkpoint_path)
else:
    print("checkpoint not found:", load_checkpoint_path)


## 13) Quick Eval After Loading

Run this right after the load cell in a fresh kernel. It uses the saved test embeddings for baseline cosine retrieval only.


In [ ]:
def _topk_cosine_torch(embeddings, query_vec, k=10, exclude_idx=None):
    emb = torch.as_tensor(embeddings, dtype=torch.float32)
    q = torch.as_tensor(query_vec, dtype=torch.float32)
    sims = emb @ q
    if exclude_idx is not None:
        sims[exclude_idx] = float("-inf")
    k = min(int(k), int(emb.shape[0] - (1 if exclude_idx is not None else 0)))
    vals, idx = torch.topk(sims, k=k, largest=True, sorted=True)
    return idx.cpu().numpy(), vals.cpu().numpy()


def _recommend(df_in, emb, qid, k=10):
    qidx = int(df_in.index[df_in["spotify_id"] == qid][0])
    idx, vals = _topk_cosine_torch(emb, emb[qidx], k=k, exclude_idx=qidx)
    out = df_in.iloc[idx].copy()
    out["similarity"] = vals
    return out


if loaded_track_ids is None or loaded_song_embeddings is None:
    print("No loaded embeddings available for quick eval.")
else:
    loaded_embedding_df = loaded_test_df[["spotify_id", "artist", "name", "tags", "year"]].copy()
    # Reorder the saved rows to match the saved embedding matrix exactly.
    loaded_embedding_df = loaded_embedding_df.set_index("spotify_id").loc[loaded_track_ids].reset_index()

    query_id = CFG["query_spotify_id"] if "CFG" in globals() else loaded_embedding_df.iloc[0]["spotify_id"]
    if query_id not in set(loaded_embedding_df["spotify_id"]):
        query_id = loaded_embedding_df.iloc[0]["spotify_id"]
        print("Configured query id not found in loaded test split; using first test track instead:", query_id)

    recs_loaded = _recommend(loaded_embedding_df, loaded_song_embeddings, query_id, k=10)
    print("query id:", query_id)
    display(recs_loaded[["artist", "name", "spotify_id", "similarity"]])
